# Project 7 - Predictive Modeling

## Objective

Build, train, evaluate, and compare multiple supervised machine learning models to predict customer churn.

### Models
- Logistic Regression
- Decision Tree
- Random Forest
- K-Nearest Neighbors
- Naive Bayes

The best-performing model is saved for later deployment.


In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

df = pd.read_csv("cleaned_data/telco_cleaned.csv")
if df.empty:
    raise ValueError("The cleaned dataset is empty.")
if "Churn" not in df.columns:
    raise ValueError("Expected target column 'Churn' was not found.")
if df["Churn"].isna().any():
    raise ValueError("Target column 'Churn' contains missing values.")
if df["Churn"].nunique() < 2:
    raise ValueError("Target column 'Churn' must contain at least two classes.")

df.head()


## Prepare Data

In [ ]:
df = df.copy()

encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

X = df.drop("Churn", axis=1)
y = df["Churn"]

class_counts = y.value_counts()
if (class_counts < 2).any():
    raise ValueError(
        "Each churn class needs at least two rows for a stratified train/test split. "
        f"Class counts: {class_counts.to_dict()}"
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)


## Define Models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "Naive Bayes": GaussianNB()
}


## Train & Evaluate

In [ ]:
results = []

best_model = None
best_accuracy = 0

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    pre = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)

    results.append([name, acc, pre, rec, f1])

    print("="*60)
    print(name)
    print(classification_report(y_test, pred))
    print("Confusion Matrix")
    print(confusion_matrix(y_test, pred))

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model

results_df = pd.DataFrame(
    results,
    columns=["Model","Accuracy","Precision","Recall","F1 Score"]
)

results_df.sort_values("Accuracy", ascending=False)


## Save Best Model

In [ ]:
import os

os.makedirs("models", exist_ok=True)

joblib.dump(best_model, "models/best_churn_model.pkl")

print("Best Accuracy:", best_accuracy)
print("Model saved to models/best_churn_model.pkl")


## Conclusion

This notebook trained multiple supervised learning models and selected the best-performing model based on accuracy. The saved model will be used in later projects for prediction and deployment.
